# E5 (2022)
[[paper]](https://arxiv.org/abs/2212.03533)<br>
E5 = <b>E</b>mb<b>E</b>ddings from bidir<b>E</b>ctional <b>E</b>ncoder r<b>E</b>presentations

E5 - попытка создать универсальный текстовый энкодер, хорошо работающий из коробки на большинстве end-to-end задач (на сопоставление или любых?)

За счет чего?
- Во-первых, акцент на сбор качественного self-supervised датасета, который назвали CCPairs. Руками много размечать дорого, нужен именно self-supervised. Почистили Common Crawl и сделали 270M пар
- Во-вторых на способе обучения - это contrastive learning на задаче сопоставления соседних кусков текста<br>(учим замечать релевантность)

<img src="../_static/img/e5/e5_1.png" width=300>

Чем другие кодировки плохи:
- разреженные представления типа BM25 не учитывают семантику
- плотные представления типа DPR (2020) учитывают, но сложно собрать хоршие hard negatives
- Contriever (2021) обучался контрастным self-supervised обучением под задачу сопосталвения текстов Inverse Cloze Task, но ему не хватало fine-tuning
- SimCSE (2021) обучался self-supervised под задачу восстановления corrupted предложений, поэтому хорошо ловил схожесть, но плохо сопоставлял

__Архитектура__<br>
В основе E5 единый BERT-подобный Encoder
1. Вход - текст с префиксом (для запроса — `query: {text}`, для документа — `passage: {text}`)
2. Единый энкодер на запрос и документ
3. Агрегация: вместо `[CLS]` токена используем Mean Pooling
4. L2-нормализация: чтобы вместо Cosine Similarity считать сразу скалярное произведение

<img src="../_static/img/e5/e5_2.png" width=300>

__Обучение__<br>
1. Contrastive Pre-training:<br> на 1.3 млрд пар с использованием InfoNCE loss. Негативы случайно из батча
2. Supervised Fine-tuning:<br> Позитивы берутся их зрамеченых датасетов (MS MARCO, NLI, Natural Questions). Негативы также майнятся (берутся Hard Negatives)

__Индексация__<br>
Все документы коллекции прогоняются через энкодер с префиксом 'passage: '. Полученные векторы сохраняются в векторную БД (например, FAISS или HNSW индекс

__Инференс__<br>
- запрос получает префикс 'query: ' и переводится в вектор тем же энкодером
- ищем K ближайших соседей (K-NN) в векторном пространстве

__Результаты__<br>
Замерялись на 56 датасетах из BEIR (Zero-shot retrieval) и MTEB:
- на 2.4 п.п. обошли Contriever по nDCG@10
- 33M модель обогнала DPR-base на 110M параметров

## Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример использования модели E5 для задачи Dense Retrieval.
# Мы будем использовать библиотеку `transformers` для загрузки предобученной модели E5.
# Для демонстрации мы создадим простую систему поиска, используя FAISS для индексации и поиска ближайших соседей.

from transformers import AutoTokenizer, AutoModel
import torch
import faiss
import numpy as np

# Загрузка предобученной модели E5 и токенизатора
model_name = "intfloat/e5-base"  # Замените на фактическое имя модели E5, если она доступна в transformers
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Функция для получения эмбеддингов текста
def get_embedding(text, prefix):
    # Добавляем префикс к тексту
    input_text = f"{prefix}: {text}"
    # Токенизация текста
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True)
    # Получение эмбеддингов
    with torch.no_grad():
        outputs = model(**inputs)
    # Используем Mean Pooling по всем токенам
    embeddings = outputs.last_hidden_state.mean(dim=1)
    # Нормализация L2
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    return embeddings.numpy()

# Пример документов и запросов
documents = [
    "Machine learning is a method of data analysis that automates analytical model building.",
    "Artificial intelligence is intelligence demonstrated by machines, in contrast to the natural intelligence displayed by humans and animals.",
    "Deep learning is part of a broader family of machine learning methods based on artificial neural networks."
]

queries = [
    "What is machine learning?",
    "Explain artificial intelligence."
]

# Получение эмбеддингов для документов
document_embeddings = np.vstack([get_embedding(doc, "passage") for doc in documents])

# Индексация документов с помощью FAISS
dimension = document_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Используем Inner Product, так как векторы нормализованы
index.add(document_embeddings)

# Обработка запросов и поиск ближайших соседей
for query in queries:
    query_embedding = get_embedding(query, "query")
    # Поиск K ближайших соседей
    K = 2
    distances, indices = index.search(query_embedding, K)
    print(f"Query: {query}")
    for i in range(K):
        print(f"Document {i+1}: {documents[indices[0][i]]} (Score: {distances[0][i]:.4f})")
    print()

# Этот пример демонстрирует использование модели E5 для задачи Dense Retrieval.
# Мы используем префиксы 'query:' и 'passage:' для различения запросов и документов,
# что помогает модели учитывать асимметрию между ними.
# FAISS используется для быстрого поиска ближайших соседей в векторном пространстве.